<a href="https://colab.research.google.com/github/carlariqc-glitch/Modelo-google-colab-gato-VS-perro/blob/main/perro_gato_v1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Fase 1: **Cargar modelo**

*   En esta fase, exportaremos el modelo de teachable machine en formato TensorFlow estándar Keras.
*   Tras descomprimirlo, lo subimos a google drive, y ahora en Python cargamos el modelo.



In [1]:
# 1. Activamos el modo legacy (Keras 2)
import os
os.environ['TF_USE_LEGACY_KERAS'] = '1'

from tensorflow import keras

#/content/drive/MyDrive/converted_keras (Unzipped Files)
mi_modelo = keras.models.load_model("/content/drive/MyDrive/modelo/keras_model.h5", compile=False)

# Carga las etiquetas de las clases
nombre_clases = open("/content/drive/MyDrive/modelo/labels.txt", "r").readlines()

print(nombre_clases)



['0 gato\n', '1 perro\n']


##Fase 2: **Adquirir datos nuevos**
Ahora, previamente cargado el modelo, cargamos una imagen, y para hacerlo usamos la librería de Python(PIL)

In [2]:
from PIL import Image, ImageOps
imagen= Image.open("/content/drive/MyDrive/perros y gatos/imagenes/test/cat.9821.jpg").convert("RGB")


## Fase 3: **Pre-procesamiento**
Ahora, cambiamos el tamaño de la imagen para que tenga 224x224 pixeles. Para esto utilizamos la librería de Numpy para que cada pixel de la imagen se represente como un número entero.


In [3]:
import numpy as np
size = (224, 224)
imagen = ImageOps.fit(imagen, size, Image.Resampling.LANCZOS)
# Convertimos la imagen en un array NumPy.
imagen_array = np.asarray(imagen)
normalizada_imagen_array = (imagen_array.astype(np.float32) / 127.5) - 1


##Fase 4: **Predicción**
Como la IA no procesa imágenes individuales, hay que transformarlas en lotes.
Tenemos que meter la imagen en un array de 1 hueco


In [4]:
# Crear un array para un lote de 1 imagen. ndarray = N-Dimensional Array
lote_imagenes = np.ndarray(shape=(1, 224, 224, 3), dtype=np.float32)
lote_imagenes[0] = normalizada_imagen_array
resultados= mi_modelo.predict(lote_imagenes)

1/1 [==============================] - 3s 3s/step


##Fase 5 :**Post-procesamiento**
En esta fase se busca el índice del número más alto en el array devuelto por el modelo (en [0.98, 0.02] sería el índice 0)
Con este índice, podemos obtener la etiqueta correspondiente de la lista de etiquetas cargada en la Fase 1 y la probabilidad asociada.




In [5]:
indice = np.argmax(resultados[0])
etiqueta = nombre_clases[indice]
print("La imagen es de clase: ", etiqueta)
probabilidad = resultados[0][indice]
print("Con una probabilidad de: ", probabilidad)

La imagen es de clase:  0 gato

Con una probabilidad de:  1.0


Con la carpeta TEST, que contiene las imágenes de prueba, guardada en google drive, creamos una variable que contenga un array de cada imagen. Para esto utilizamos la librería os de Python.

In [6]:
lista_archivos = os.listdir("/content/drive/MyDrive/perros y gatos/imagenes/test")


In [7]:
total_predicciones = 0
aciertos = 0
media_probabilidad_aciertos = 0.0
# predicciones incorrectas contendrá una lista con los nombres de las imágenes mal clasificadas
predicciones_incorrectas = []

In [8]:
def predecir_imagen(rutaimg):
    imagen = Image.open(rutaimg).convert("RGB")
